In [1]:
import rdkit
import pandas as pd
from validation import *
from rdkit.Chem import Descriptors, Crippen, Lipinski
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
import numpy as np

In [25]:
lst_quants = ['BF16', 'Q8', 'Q6', 'Q4']
lst_models = ['Mistral4-119B', 'Qwen3.6-35B', 'gemma-4-31B', 'gpt-oss-120B']
lst_replicas = range(1,20+1)

# TASK 1

In [26]:
path = '/home/leandro/repos/gcrm/csv_llms_task_1'

In [27]:
d = {}
for m in lst_models:
    for q in lst_quants:
        for r in lst_replicas:
            lst_smi = pd.read_csv(f'{path}/{m}-{q}_replica_{r}.csv')['SMILES'].values.tolist()
            n_smi_total = len(lst_smi)
            lst_valid_smi = validate_smiles(lst_smi)
            n_smi_valid = len(lst_valid_smi)
                
            print(f'{m:15s} {q:5s} {r:2.0f} {n_smi_total:2d} {n_smi_valid:2d}')
            d[f'{m}-{q}-{r}'] = { 'm':m, 'q':q, 'r':r, 'n_smi_total':n_smi_total, 'n_smi_valid':n_smi_valid}

Mistral4-119B   BF16   1 20 19
Mistral4-119B   BF16   2 19 15
Mistral4-119B   BF16   3 19 16
Mistral4-119B   BF16   4 19 19
Mistral4-119B   BF16   5 19 19
Mistral4-119B   BF16   6 20 17
Mistral4-119B   BF16   7  0  0
Mistral4-119B   BF16   8 19 15
Mistral4-119B   BF16   9 19 16
Mistral4-119B   BF16  10 20 20
Mistral4-119B   BF16  11 20 17
Mistral4-119B   BF16  12 21 19
Mistral4-119B   BF16  13 20 16
Mistral4-119B   BF16  14 20 19
Mistral4-119B   BF16  15  0  0
Mistral4-119B   BF16  16 20 14
Mistral4-119B   BF16  17 20 16
Mistral4-119B   BF16  18 19 18
Mistral4-119B   BF16  19 20 15
Mistral4-119B   BF16  20 15 12
Mistral4-119B   Q8     1 20 17
Mistral4-119B   Q8     2  0  0
Mistral4-119B   Q8     3 19 19
Mistral4-119B   Q8     4 20 20
Mistral4-119B   Q8     5 22 18
Mistral4-119B   Q8     6 19 16
Mistral4-119B   Q8     7 22 18
Mistral4-119B   Q8     8 20 20
Mistral4-119B   Q8     9 20 20
Mistral4-119B   Q8    10  1  1
Mistral4-119B   Q8    11 20 17
Mistral4-119B   Q8    12 19 19
Mistral4

In [28]:
df = pd.DataFrame.from_dict(d, orient='index')
df['pct_smi_valid'] = df['n_smi_valid']/df['n_smi_total']
df

,m,q,r,n_smi_total,n_smi_valid,pct_smi_valid
Mistral4-119B-BF16-1,Mistral4-119B,BF16,1,20,19,0.950000
Mistral4-119B-BF16-2,Mistral4-119B,BF16,2,19,15,0.789474
Mistral4-119B-BF16-3,Mistral4-119B,BF16,3,19,16,0.842105
Mistral4-119B-BF16-4,Mistral4-119B,BF16,4,19,19,1.000000
Mistral4-119B-BF16-5,Mistral4-119B,BF16,5,19,19,1.000000
...,...,...,...,...,...,...
gpt-oss-120B-Q4-16,gpt-oss-120B,Q4,16,20,19,0.950000
gpt-oss-120B-Q4-17,gpt-oss-120B,Q4,17,20,19,0.950000
gpt-oss-120B-Q4-18,gpt-oss-120B,Q4,18,20,20,1.000000
gpt-oss-120B-Q4-19,gpt-oss-120B,Q4,19,20,20,1.000000


In [29]:
d_stats = {}
for m in lst_models:
    d_stats[m] = {}
    for q in lst_quants:
        d_stats[m][q] = {}
        
        avg = df.loc[(df['m']==m) & (df['q']==q)]['pct_smi_valid'].mean()
        std = df.loc[(df['m']==m) & (df['q']==q)]['pct_smi_valid'].std()
        d_stats[m][q]['pct_smi_valid'] = (float(avg), float(std))
        

In [30]:
print('-'*39)
print(f'|        MODEL       | pct_smi_valid  |')
print('-'*39)

for m in lst_models:
    for q in lst_quants:
        valid_avg, valid_std = d_stats[m][q]['pct_smi_valid']
        
        print(f'| {m:13s} {q:4s} | {valid_avg*100:5.2f} +- {valid_std*100:5.2f} |')
    print(f'|                    |                |')
print('-'*71)


---------------------------------------
|        MODEL       | pct_smi_valid  |
---------------------------------------
| Mistral4-119B BF16 | 86.47 +-  9.19 |
| Mistral4-119B Q8   | 90.03 +- 13.92 |
| Mistral4-119B Q6   | 87.72 +-  9.61 |
| Mistral4-119B Q4   | 67.57 +- 29.80 |
|                    |                |
| Qwen3.6-35B   BF16 | 95.59 +-  5.58 |
| Qwen3.6-35B   Q8   | 93.75 +-  6.26 |
| Qwen3.6-35B   Q6   | 97.75 +-  3.80 |
| Qwen3.6-35B   Q4   | 95.00 +-  6.69 |
|                    |                |
| gemma-4-31B   BF16 | 90.25 +-  6.58 |
| gemma-4-31B   Q8   | 87.50 +- 10.70 |
| gemma-4-31B   Q6   | 90.75 +-  6.74 |
| gemma-4-31B   Q4   | 92.75 +-  9.66 |
|                    |                |
| gpt-oss-120B  BF16 | 94.00 +-  5.53 |
| gpt-oss-120B  Q8   | 91.75 +-  8.93 |
| gpt-oss-120B  Q6   | 95.50 +-  4.84 |
| gpt-oss-120B  Q4   | 97.00 +-  2.99 |
|                    |                |
-----------------------------------------------------------------------


In [31]:
def n_lipinski_violations(smiles_list, threshold=1):
    n_smi_viol = 0
    n_violations = 0
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        violations = sum([
            Descriptors.MolWt(mol) > 500,
            Crippen.MolLogP(mol) > 5,
            Lipinski.NumHDonors(mol) > 5,
            Lipinski.NumHAcceptors(mol) > 10
        ])
        n_violations += violations

        if violations >= threshold:
            n_smi_viol+=1
    return n_violations, n_smi_viol

# TASK 2

In [32]:
path = '/home/leandro/repos/gcrm/csv_llms_task_2'

In [33]:
m = lst_models[-1]
r = 1
q = 'BF16'
lst_smi = pd.read_csv(f'{path}/{m}-{q}_replica_{r}.csv')['SMILES'].values.tolist()
lst_valid_smi = validate_smiles(lst_smi)
n_lipinski_violations(lst_valid_smi, threshold=1)

(13, 11)

In [34]:
d = {}
for m in lst_models:
    for q in lst_quants:
        for r in lst_replicas:
            lst_smi = pd.read_csv(f'{path}/{m}-{q}_replica_{r}.csv')['SMILES'].values.tolist()
            n_smi_total = len(lst_smi)
            lst_valid_smi = validate_smiles(lst_smi)
            n_smi_valid = len(lst_valid_smi)
            
            if n_smi_valid == 0:
                n_smi_violations = np.nan
                n_tot_violations = np.nan
                intdiv = np.nan
            elif n_smi_valid == 1:
                n_tot_violations, n_smi_violations = n_lipinski_violations(lst_valid_smi, threshold=1)
                
                intdiv = np.nan
            else:
                n_tot_violations, n_smi_violations = n_lipinski_violations(lst_valid_smi, threshold=1)
                intdiv = calculate_intdiv(lst_valid_smi)
                
            print(f'{m:15s} {q:5s} {r:2.0f} {n_smi_total:2d} {n_smi_valid:2d} {n_tot_violations:2.0f} {n_smi_violations:4.0f} {intdiv:5.2f}')
            d[f'{m}-{q}-{r}'] = { 'm':m, 'q':q, 'r':r, 'n_smi_total':n_smi_total, 'n_smi_valid':n_smi_valid,
                                  'n_tot_violations':n_tot_violations, 'n_smi_violations':n_smi_violations,
                                  'intdiv':intdiv}

Mistral4-119B   BF16   1 19  8  4    4  0.75
Mistral4-119B   BF16   2 19  3  0    0  0.83
Mistral4-119B   BF16   3  0  0 nan  nan   nan
Mistral4-119B   BF16   4 19 18  0    0  0.74
Mistral4-119B   BF16   5 19  4  1    1  0.86
Mistral4-119B   BF16   6 20 17  0    0  0.86
Mistral4-119B   BF16   7 19 18  2    2  0.84
Mistral4-119B   BF16   8 19 19  0    0  0.81
Mistral4-119B   BF16   9 19 16  0    0  0.68
Mistral4-119B   BF16  10 19 13  0    0  0.83
Mistral4-119B   BF16  11 19 15  1    1  0.87
Mistral4-119B   BF16  12  0  0 nan  nan   nan
Mistral4-119B   BF16  13 30 17  1    1  0.85
Mistral4-119B   BF16  14 19 15  0    0  0.68
Mistral4-119B   BF16  15 18 17  0    0  0.65
Mistral4-119B   BF16  16 19 16  1    1  0.83
Mistral4-119B   BF16  17 19 12  0    0  0.81
Mistral4-119B   BF16  18 20 15  2    2  0.79
Mistral4-119B   BF16  19 21 17  0    0  0.74
Mistral4-119B   BF16  20 21 21  0    0  0.83
Mistral4-119B   Q8     1 19 15  0    0  0.83
Mistral4-119B   Q8     2  1  1  0    0   nan
Mistral4

In [35]:
df = pd.DataFrame.from_dict(d, orient='index')
df['pct_smi_valid'] = df['n_smi_valid']/df['n_smi_total']
df['pct_lipinski_viol'] = df['n_smi_violations']/df['n_smi_valid']
df['avg_viol_per_smi'] = df['n_tot_violations']/df['n_smi_valid']
df

,m,q,r,n_smi_total,n_smi_valid,n_tot_violations,n_smi_violations,intdiv,pct_smi_valid,pct_lipinski_viol,avg_viol_per_smi
Mistral4-119B-BF16-1,Mistral4-119B,BF16,1,19,8,4.0,4.0,0.751562,0.421053,0.50,0.50
Mistral4-119B-BF16-2,Mistral4-119B,BF16,2,19,3,0.0,0.0,0.827236,0.157895,0.00,0.00
Mistral4-119B-BF16-3,Mistral4-119B,BF16,3,0,0,NaN,NaN,NaN,NaN,NaN,NaN
Mistral4-119B-BF16-4,Mistral4-119B,BF16,4,19,18,0.0,0.0,0.739384,0.947368,0.00,0.00
Mistral4-119B-BF16-5,Mistral4-119B,BF16,5,19,4,1.0,1.0,0.860368,0.210526,0.25,0.25
...,...,...,...,...,...,...,...,...,...,...,...
gpt-oss-120B-Q4-16,gpt-oss-120B,Q4,16,20,15,0.0,0.0,0.681385,0.750000,0.00,0.00
gpt-oss-120B-Q4-17,gpt-oss-120B,Q4,17,20,15,0.0,0.0,0.749012,0.750000,0.00,0.00
gpt-oss-120B-Q4-18,gpt-oss-120B,Q4,18,20,14,0.0,0.0,0.594732,0.700000,0.00,0.00
gpt-oss-120B-Q4-19,gpt-oss-120B,Q4,19,20,19,0.0,0.0,0.824712,0.950000,0.00,0.00


In [36]:
d_stats = {}
for m in lst_models:
    d_stats[m] = {}
    for q in lst_quants:
        d_stats[m][q] = {}
        avg = df.loc[(df['m']==m) & (df['q']==q)]['pct_lipinski_viol'].mean()
        std = df.loc[(df['m']==m) & (df['q']==q)]['pct_lipinski_viol'].std()
        d_stats[m][q]['pct_lipinski_viol'] = (float(avg), float(std))

        avg = df.loc[(df['m']==m) & (df['q']==q)]['intdiv'].mean()
        std = df.loc[(df['m']==m) & (df['q']==q)]['intdiv'].std()
        d_stats[m][q]['intdiv'] = (float(avg), float(std))

        avg = df.loc[(df['m']==m) & (df['q']==q)]['pct_smi_valid'].mean()
        std = df.loc[(df['m']==m) & (df['q']==q)]['pct_smi_valid'].std()
        d_stats[m][q]['pct_smi_valid'] = (float(avg), float(std))

        avg = df.loc[(df['m']==m) & (df['q']==q)]['avg_viol_per_smi'].mean()
        std = df.loc[(df['m']==m) & (df['q']==q)]['avg_viol_per_smi'].std()
        d_stats[m][q]['avg_viol_per_smi'] = (float(avg), float(std))
        

In [37]:
print('-'*86)
print(f'|        MODEL       | pct_smi_valid  | pct_lipin_viol | avg_viol/smi | intdiv       |')
print('-'*86)

for m in lst_models:
    for q in lst_quants:
        valid_avg, valid_std = d_stats[m][q]['pct_smi_valid']
        viol_avg, viol_std = d_stats[m][q]['pct_lipinski_viol']
        viol_per_smi_avg, viol_per_smi_std = d_stats[m][q]['avg_viol_per_smi']
        intdiv_avg, intdiv_std = d_stats[m][q]['intdiv']
        
        print(f'| {m:13s} {q:4s} | \
{valid_avg*100:5.2f} +- {valid_std*100:5.2f} | \
{viol_avg*100:5.2f} +- {viol_std*100:5.2f} | \
{viol_per_smi_avg:4.2f} +- {viol_per_smi_std:4.2f} | \
{intdiv_avg:4.2f} +- {intdiv_std:4.2f} | ')
    print(f'|                    |                |                |              |              |')
print('-'*86)


--------------------------------------------------------------------------------------
|        MODEL       | pct_smi_valid  | pct_lipin_viol | avg_viol/smi | intdiv       |
--------------------------------------------------------------------------------------
| Mistral4-119B BF16 | 73.24 +- 25.09 |  6.57 +- 12.76 | 0.07 +- 0.13 | 0.79 +- 0.07 | 
| Mistral4-119B Q8   | 67.34 +- 33.06 | 13.84 +- 31.29 | 0.14 +- 0.31 | 0.80 +- 0.06 | 
| Mistral4-119B Q6   | 53.29 +- 25.22 |  6.41 +- 10.75 | 0.06 +- 0.11 | 0.80 +- 0.04 | 
| Mistral4-119B Q4   | 44.12 +- 30.14 |  3.07 +-  6.87 | 0.04 +- 0.09 | 0.82 +- 0.03 | 
|                    |                |                |              |              |
| Qwen3.6-35B   BF16 | 81.75 +- 14.07 | 32.33 +- 26.23 | 0.43 +- 0.37 | 0.74 +- 0.05 | 
| Qwen3.6-35B   Q8   | 82.00 +- 18.17 | 25.29 +- 25.88 | 0.33 +- 0.38 | 0.72 +- 0.08 | 
| Qwen3.6-35B   Q6   | 68.25 +- 29.70 | 26.04 +- 23.35 | 0.33 +- 0.34 | 0.68 +- 0.11 | 
| Qwen3.6-35B   Q4   | 71.91 +- 23.9

In [38]:
n_lipinski_violations(lst_valid_smi, threshold=1)

(0, 0)

In [39]:

def calculate_intdiv(smiles_list):
    lst_mol = [Chem.MolFromSmiles(smi) for smi in smiles_list]
    lst_fps = [AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048) for mol in lst_mol]
    total = len(lst_mol)

    # Tanimoto Similarity for all unique pairs
    lst_sim = []
    
    for i in range(total):
        for j in range(i + 1, total):
            sim = DataStructs.TanimotoSimilarity(lst_fps[i], lst_fps[j])
            lst_sim.append(sim)
            
    # 3. Diversidade = 1 - Média das Similaridades
    avg_sim = sum(lst_sim) / len(lst_sim)
    int_div = 1.0 - avg_sim
    
    return int_div

In [40]:
intdiv(lst_valid_smi)

TypeError: 'float' object is not callable